# Hologemma tutor: fine-tune Gemma on Kaggle

Turns Gemma 4 E4B (instruction-tuned) into the Hologemma lab tutor: it answers questions about the part a student selected
(heart and computer models), in English or Mongolian, grounded in the facts in the system prompt.

**Before you run**
1. *Settings → Accelerator*: **GPU T4 x2** (or P100). *Settings → Internet*: **on**.
2. *Add Input → Datasets*: upload `training/data/` from the repo (`train.jsonl`, `eval.jsonl`) as a dataset named **hologemma-tutor-data**.
3. *Add-ons → Secrets*: add **HF_TOKEN** (a Hugging Face write token). Accept the Gemma licence on Hugging Face with the same account.
4. Run all. Training takes roughly 30–60 minutes on a T4.

The outputs (`/kaggle/working`): the merged model pushed to Hugging Face, a GGUF copy, and `eval_outputs.jsonl`
with base vs fine-tuned answers. Score that file locally with `npx tsx scripts/judge-tutor-eval.ts`.

In [ ]:
%%capture
!pip install -U unsloth

In [ ]:
import glob, json, os, random
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:  # secret not attached to this notebook: train anyway, push the saved model later
    print("No HF_TOKEN secret, skipping the Hugging Face push:", e)
    HF_TOKEN = None
DATA = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])

BASE_MODEL = "unsloth/gemma-4-E4B-it"  # Gemma 4 E4B: QLoRA needs about 10 GB, fits a T4
HF_REPO = "berkhw/hologemma-tutor"  # private repo the merged model is pushed to
MAX_SEQ = 2048
EPOCHS = 5  # a small demo set (about 80 rows): more passes so the answers stick
print("data:", DATA)

## Load Gemma in 4-bit and add LoRA adapters

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    token=HF_TOKEN,  # None is fine: the Unsloth copy of Gemma is not gated
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,  # text only
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

## Data

Each row is the exact chat the app sends: system prompt (role, language, selected part's facts), the student's
question(s), and the tutor's answer(s). Gemma 4's template has a system role, so the prompt stays as the app sends it.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
data = load_dataset("json", data_files={"train": f"{DATA}/train.jsonl", "eval": f"{DATA}/eval.jsonl"})

def to_text(batch):
    texts = [tokenizer.apply_chat_template(m, tokenize=False).removeprefix("<bos>") for m in batch["messages"]]
    return {"text": texts}

train = data["train"].map(to_text, batched=True)
print(len(train), "train rows")
print(train[0]["text"][:1500])
assert "<|turn>model\n" in train[0]["text"], "response marker missing: the loss would be empty"

## Train (loss on the tutor's answers only)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        seed=3407,
        report_to="none",
        output_dir="/kaggle/working/checkpoints",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)
stats = trainer.train()
print(stats)

## Before / after on held-out questions

The same eval questions go to the base model (adapters switched off) and to the fine-tuned model.

In [ ]:
import torch

def answer(messages, max_new_tokens=250):
    # Gemma 4's tokenizer is a multimodal processor: the text must be passed by name, or it is read as an image.
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False).removeprefix("<bos>")
    inputs = tokenizer(text=[text], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=1.0, top_p=0.95, top_k=64)
    return tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

random.seed(0)
rows = list(data["eval"])
sample = random.sample(rows, min(100, len(rows)))

FastModel.for_inference(model)
results = []
for i, row in enumerate(sample):
    prompt = row["messages"][:-1]
    with model.disable_adapter():
        base = answer(prompt)
    tuned = answer(prompt)
    results.append({
        "model": row["model"], "part": row["part"], "lang": row["lang"], "kind": row["kind"],
        "messages": prompt, "reference": row["messages"][-1]["content"], "base": base, "tuned": tuned,
    })
    if i % 20 == 0:
        print(i, row["lang"], "|", prompt[-1]["content"], "\n  base: ", base[:200], "\n  tuned:", tuned[:200])

with open("/kaggle/working/eval_outputs.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("saved", len(results))

## Demo questions

One trained question per model and language, answered by base and fine-tuned Gemma: the ones to ask on stage.


In [ ]:
demo = {}
for row in data["train"]:
    demo.setdefault((row["model"], row["lang"]), row)
with open("/kaggle/working/demo_outputs.jsonl", "w") as f:
    for (model_id, lang), row in sorted(demo.items()):
        prompt = row["messages"][:-1]
        with model.disable_adapter():
            base = answer(prompt)
        tuned = answer(prompt)
        print(f"{model_id}/{row['part']}/{lang} | {prompt[-1]['content']}\n  base:  {base}\n  tuned: {tuned}\n")
        f.write(json.dumps({"model": model_id, "part": row["part"], "lang": lang, "question": prompt[-1]["content"],
                            "reference": row["messages"][-1]["content"], "base": base, "tuned": tuned}, ensure_ascii=False) + "\n")

## Try it

In [ ]:
sys_prompt = sample[0]["messages"][0]["content"]
for q in ["where does this plug in?", "энэ юу хийдэг вэ?"]:
    print(q, "->", answer([{"role": "system", "content": sys_prompt}, {"role": "user", "content": q}]), "\n")

## Save

The merged 16-bit model goes to a private Hugging Face repo (served by an Inference Endpoint), and a Q8 GGUF copy is
kept in `/kaggle/working` as an offline backup for Ollama / llama.cpp.

In [ ]:
model.save_pretrained("/kaggle/working/lora")  # adapters only, small
tokenizer.save_pretrained("/kaggle/working/lora")
if HF_TOKEN:
    model.push_to_hub_merged(HF_REPO, tokenizer, token=HF_TOKEN, private=True)
else:
    model.save_pretrained_merged("/kaggle/working/merged", tokenizer)
    print("Saved the merged model to /kaggle/working/merged; push it to Hugging Face from there.")

In [ ]:
# Offline backup for Ollama. Building llama.cpp can fail on some images; the model above is already saved.
try:
    model.save_pretrained_gguf("/kaggle/working/gguf", tokenizer, quantization_type="Q8_0")
except Exception as e:
    print("GGUF export failed, skipping:", e)
!ls -lh /kaggle/working/gguf 2>/dev/null